
# Data Pipeline: 
FMCG Fact Sales Ingestion & Transformation (Medallion Architecture)This notebook implements an End-to-End Medallion Architecture (Bronze -> Silver -> Gold) data ingestion and transformation pipeline using PySpark and Delta Lake on Databricks.

In [0]:
# Import PySpark window functions, standard functions, and Delta table modules
from pyspark.sql import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "../Utitlites"

In [0]:
# ==========================================
# 2. Dynamic Notebook Parameters (Widgets)
# ==========================================
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_Sourse', 'customers', 'Data Source')

# Extract parameter values
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_Sourse')

In [0]:
# Read raw CSV file and attach metadata attributes
df = (spark.read.format('csv')
        .option('header', True)
        .option('inferSchema', True)
        .load(f'{sales_land_path}/*.csv')
        .withColumn('read_timestamp', F.current_timestamp())
        .select('*', '_metadata.file_name', '_metadata.file_size')
)

# Display Schema
df.printSchema()

# Dispay Sample
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FOCT62720602|Tuesday, Septembe...|     ABC987|  25891301|     71.0|2026-10-08 09:41:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891502|    125.0|2026-10-08 09:41:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|   

## Bronze Layer
- Reads raw CSV files from the processed storage path, enriches records with ingestion metadata (read_timestamp, file_name, file_size), and overwrites the Bronze Delta table with Change Data Feed (CDF) enabled.

In [0]:
# Process Data For Bronze Layer
df.write \
    .format('delta') \
    .mode('overwrite') \
    .option('delta.enableChangeDataFeed', True) \
    .saveAsTable(f'{catalog}.bronze.sb_fact_{data_source}') \

# Show Results
spark.sql(f'SELECT * FROM {catalog}.bronze.sb_fact_{data_source}').show()        

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-10-08 09:41:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-10-08 09:41:...|orders_2025_07_01...|    20744|
|FJUL33320501|          01-07-2025|     789320|  25891403|     NULL|2026-10-08 09:41:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891201|    354.0|2026-10-08 09:41:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891501|    249.0|2026-10-08 09:41:...|orders_2025_07_01...|    20744|
|FJUL33320501|          01-07-20

In [0]:
df = spark.read.table(f'{catalog}.bronze.sb_fact_{data_source}')
df = df.filter(F.col("order_id").isNotNull())

# Show Results
df.createOrReplaceTempView("df")
spark.sql('SELECT order_id FROM df WHERE order_id IS NULL').show()

+--------+
|order_id|
+--------+
+--------+



### Data Cleaning

In [0]:
# 1. Target entity columns requiring sanitization
entity_cols = ['product_id', 'customer_id']

# 2. Sanitize entity identifiers using regular expressions
for col_name in entity_cols:
    df = df.withColumn(
        col_name,
        F.when(F.col(col_name).rlike('^[0-9]+$'), F.col(col_name))
         .otherwise('999999')
    )

# 3. Verify distinct sanitized keys
df.select(['customer_id', 'product_id']).distinct().show()

# 4. Display log of repaired records
print('\nRepaired Records :\n')
df.filter((F.col('customer_id') == '999999') | (F.col('product_id') == '999999')) \
  .groupBy('customer_id', 'product_id') \
  .count() \
  .show()

+-----------+----------+
|customer_id|product_id|
+-----------+----------+
|     789320|  25891203|
|     789320|  25891301|
|     789320|  25891403|
|     789320|  25891201|
|     789320|  25891501|
|     789401|  25891302|
|     789401|  25891502|
|     789401|  25891503|
|     789401|  25891203|
|     789401|  25891201|
|     789401|  25891603|
|     789101|  25891503|
|     789101|  25891601|
|     789101|  25891501|
|     789101|  25891101|
|     789101|  25891303|
|     789202|  25891403|
|     789202|  25891103|
|     789202|  25891601|
|     789202|  25891603|
+-----------+----------+
only showing top 20 rows

Repaired Records :

+-----------+----------+-----+
|customer_id|product_id|count|
+-----------+----------+-----+
|     999999|  25891602|  306|
|     999999|  25891202|  305|
|     999999|  25891401|  256|
|     999999|  25891101|  244|
|     999999|  25891503|  250|
|     999999|  25891402|  283|
|     999999|  25891501|  291|
|     999999|  25891103|  291|
|     999999|

### Complex Date Parsing Standardisation
Normalizes string-formatted dates across multiple irregular patterns into standard Spark date types.

In [0]:
# 1. Enable legacy time parser policy for backwards compatibility
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# 2. String cleaning and delimiter normalization
df = df.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# Replace textual month names with numeric equivalents
months_map = {
    "January": "1", "February": "2", "March": "3", "April": "4",
    "May": "5", "June": "6", "July": "7", "August": "8",
    "September": "9", "October": "10", "November": "11", "December": "12"
}

for month_name, month_num in months_map.items():
    df = df.withColumn(
        "order_placement_date",
        F.regexp_replace(F.col("order_placement_date"), month_name, month_num)
    )

# Standardize delimiters to slashes
df = df.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"\s*,\s*", "/")
)
df = df.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"\s+", "/")
)

# 3. Convert multi-pattern date strings using try_to_date fallback chain
df = df.withColumn(
    "order_placement_date",
    F.coalesce(
        F.expr("try_to_date(order_placement_date, 'M/d/yyyy')"),
        F.expr("try_to_date(order_placement_date, 'd/M/yyyy')"),
        F.expr("try_to_date(order_placement_date, 'yyyy/M/d')"),
        F.expr("try_to_date(order_placement_date, 'd-M-yyyy')"),
        F.expr("try_to_date(order_placement_date, 'yyyy-M-d')")
    )
)

# Identify invalid or unparseable date records
invalid_or_null_df = df.filter(
    F.col("order_placement_date").isNull() | 
    (~F.col("order_placement_date").rlike(r"^\d{4}-\d{2}-\d{2}$"))
)

# Display unparseable distinct entries for QA evaluation
invalid_or_null_df.select("order_placement_date").distinct().show(truncate=False)

+--------------------+
|order_placement_date|
+--------------------+
+--------------------+



In [0]:
# Drop records missing order quantities
df = df.filter(F.col('order_qty').isNotNull())
df.select("order_qty").filter(F.col("order_qty").isNull()).show()

# Remove duplicate entries
df = df.dropDuplicates()    

# Output temporal coverage bounds
df.agg(
    F.min('order_placement_date').alias('Min Date'),
    F.max('order_placement_date').alias('Max Date')
).show()

+---------+
|order_qty|
+---------+
+---------+

+----------+----------+
|  Min Date|  Max Date|
+----------+----------+
|2025-01-07|2025-12-11|
+----------+----------+



In [0]:
# Fetch dimension metadata from Gold layer
df_products = spark.sql(f'SELECT * FROM {catalog}.gold.sb_dim_products')

# Drop Duplicates
df = df.dropDuplicates(subset=['order_id', 'product_id', 'customer_id'])

# Perform Inner Join to map business product codes
df = df.join(df_products,
             on='product_id',
             how='inner'
             ).select(df['*'], df_products['product_code'])

df.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|        product_code|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
|FJUL32101601|          2025-07-01|     789101|  25891601|    124.0|2026-10-08 09:41:...|orders_2025_07_01...|    20744|716fa4e54b7894c91...|
|FJUL32221603|          2025-07-01|     789221|  25891501|    140.0|2026-10-08 09:41:...|orders_2025_07_01...|    20744|ee1f7df9cf660ef02...|
|FJUL33321602|          2025-07-01|     789321|  25891101|    345.0|2026-10-08 09:41:...|orders_2025_07_01...|    20744|e91ba9d665f90254d...|
|FJUL33503502|          2025-07-01|     789503|  25891101|    362.0|2026-10-08 09:41:...|orders_2025_07_01...|    20744|e91ba9d665f90254d...|
|FJUL3

## Silver Layer Upsert

In [0]:
# Target Silver Table Name
table_name = f'{catalog}.silver.sb_fact_{data_source}'

# Check table existence and perform initial write or MERGE operation
if not (spark.catalog.tableExists(table_name)):
    df.write \
        .format('delta') \
        .mode('overwrite') \
        .option('delta.enableChangeDataFeed', 'true') \
        .option('mergeSchema', 'true') \
        .saveAsTable(table_name)
else:
    silver_fact_orders = DeltaTable.forName(spark, table_name)
    
    silver_fact_orders.alias('target').merge(
        source=df.alias('source'),
        condition='''
            target.order_placement_date = source.order_placement_date 
            AND target.order_id = source.order_id 
            AND target.product_code = source.product_code 
            AND target.customer_id = source.customer_id
        '''
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

# Display updated Silver dataset
spark.sql(f'SELECT * FROM {table_name}').show()

+-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
|     order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|        product_code|
+-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
|FNOV731521201|          2025-11-30|     789521|  25891201|    321.0|2026-10-08 09:41:...|orders_2025_11_30...|    41220|ac8b5d1e6a188f56b...|
| FDEC83721202|          2025-11-30|     789721|  25891202|    340.0|2026-10-08 09:41:...|orders_2025_11_30...|    41220|96b71be076213a8ec...|
| FDEC83520601|          2025-11-30|     789520|  25891601|     76.0|2026-10-08 09:41:...|orders_2025_11_30...|    41220|716fa4e54b7894c91...|
| FDEC83220203|          2025-11-30|     789220|  25891203|    367.0|2026-10-08 09:41:...|orders_2025_11_30...|    41220|91a67435e55806eae...|

## Gold Processing

### Upload for gold sb_fact_orders

In [0]:
# Transform Silver schema into Gold dimensional attribute standards
df_gold = spark.sql(f'''
    SELECT  order_id,
            order_placement_date AS date,
            product_code,
            customer_id AS customer_code,
            order_qty AS sold_quantity
    FROM {catalog}.silver.sb_fact_{data_source}
''')

# Target Gold Fact Table Name
table_name = f'{catalog}.gold.sb_fact_{data_source}'

# Create or Merge into Gold Table
if not (spark.catalog.tableExists(table_name)):
    df_gold.write \
        .format('delta') \
        .mode('overwrite') \
        .option('delta.enableChangeDataFeed', 'true') \
        .option('mergeSchema', 'true') \
        .saveAsTable(table_name)
else:
    gold_fact_orders = DeltaTable.forName(spark, table_name)
    
    # Bug Fix Note: Using gold_fact_orders as target and df_gold as source
    gold_fact_orders.alias('target').merge(
        source=df_gold.alias('source'),
        condition='''
            target.date = source.date 
            AND target.order_id = source.order_id 
            AND target.product_code = source.product_code 
            AND target.customer_code = source.customer_code
        '''
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

spark.sql(f'SELECT * FROM {table_name}').show()

+------------+----------+--------------------+-------------+-------------+
|    order_id|      date|        product_code|customer_code|sold_quantity|
+------------+----------+--------------------+-------------+-------------+
|FJUL32101601|2025-07-01|716fa4e54b7894c91...|       789101|        124.0|
|FJUL32221603|2025-07-01|ee1f7df9cf660ef02...|       789221|        140.0|
|FJUL33321602|2025-07-01|e91ba9d665f90254d...|       789321|        345.0|
|FJUL33503502|2025-07-01|e91ba9d665f90254d...|       789503|        362.0|
|FJUL34622602|2025-07-01|716fa4e54b7894c91...|       789622|         91.0|
|FJUL33202502|2025-07-01|e91ba9d665f90254d...|       789202|        391.0|
|FJUL33501401|2025-07-01|e91ba9d665f90254d...|       999999|        484.0|
|FJUL33420203|2025-07-01|91a67435e55806eae...|       789420|        138.0|
|FJUL34103403|2025-07-01|77b6f538a9d0e0cf8...|       789103|        420.0|
|FJUL32102602|2025-07-01|778c2a7aa27bfdb21...|       789102|         63.0|
|FJUL32422401|2025-07-01|

### Monthly Aggregation & Parent Gold Fact Table Upsert

In [0]:
# Truncate dates to monthly grain and aggregate total quantities
df_month_start = (
    df_gold
    .withColumn('month_start', F.trunc('date', 'MM'))
    .groupBy(['month_start', 'customer_code', 'product_code'])
    .agg(F.sum('sold_quantity').alias('sold_quantity'))
    .withColumnRenamed('month_start', 'date')
)

df_month_start.show()

# Drop Duplicates From df_month_start
df_month_start = df_month_start.dropDuplicates(subset=['product_code', 'date', 'customer_code'])

# Reference Parent Consolidated Gold Fact Table
df_sales_parent = DeltaTable.forName(spark, f'{catalog}.gold.fact_{data_source}')

# Merge aggregated monthly facts into Parent Gold table
(
    df_sales_parent.alias('target')
    .merge(
        source=df_month_start.alias('source'),
        condition='''
            target.product_code = source.product_code
            AND target.customer_code = source.customer_code
            AND target.date = source.date
        '''
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

+----------+-------------+--------------------+-------------+
|      date|customer_code|        product_code|sold_quantity|
+----------+-------------+--------------------+-------------+
|2025-11-01|       789402|102628255d24304d6...|       3795.0|
|2025-08-01|       789321|96b71be076213a8ec...|       2499.0|
|2025-01-01|       789520|062f5574bbdf4386b...|        112.0|
|2025-10-01|       789401|3cab59f0592428527...|        576.0|
|2025-09-01|       999999|77b6f538a9d0e0cf8...|      18553.0|
|2025-03-01|       789321|fe5a8036be4b9a787...|        392.0|
|2025-08-01|       999999|77b6f538a9d0e0cf8...|      17708.0|
|2025-07-01|       999999|fe5a8036be4b9a787...|      18347.0|
|2025-02-01|       789422|d9ebd1ca64d23951a...|         35.0|
|2025-05-01|       999999|102628255d24304d6...|        417.0|
|2025-10-01|       789101|716fa4e54b7894c91...|       1376.0|
|2025-09-01|       789121|062f5574bbdf4386b...|       1896.0|
|2025-08-01|       789520|102628255d24304d6...|       3359.0|
|2025-12

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## Check Quality

In [0]:
print("=" * 70)
print("STARTING FINAL QUALITY AUDIT FOR GOLD LAYER TABLES")
print("=" * 70)

# Read directly from Gold Catalog Tables
gold_sb_fact = spark.read.table(f'{catalog}.gold.sb_fact_{data_source}')
gold_parent_fact = spark.read.table(f'{catalog}.gold.fact_{data_source}')

# AUDIT PART A: Detailed Gold Fact Table
sb_keys = ["order_id", "product_code", "customer_code"]
sb_total = gold_sb_fact.count()
sb_nulls = gold_sb_fact.filter(
    F.col("order_id").isNull() | F.col("product_code").isNull() | F.col("customer_code").isNull()
).count()
sb_duplicates = gold_sb_fact.groupBy(sb_keys).count().filter(F.col("count") > 1).count()

print(f"\n[AUDIT 1] Detailed Gold Fact Table (gold.sb_fact_{data_source}):")
print(f"  ├─ Total Gold Records: {sb_total}")
print(f"  ├─ Null Key Components: {sb_nulls}")
print(f"  └─ True Duplicate Composite Keys: {sb_duplicates}")

if sb_duplicates > 0:
    print("WARNING: Duplicates detected in Detailed Gold Table!")
    gold_sb_fact.groupBy(sb_keys).count().filter(F.col("count") > 1).show(5)
else:
    print("PASSED: Detailed Gold Table is 100% clean and unique.")

# AUDIT PART B: Monthly Aggregated Parent Fact Table
parent_keys = ["product_code", "customer_code", "date"]
parent_total = gold_parent_fact.count()
parent_nulls = gold_parent_fact.filter(
    F.col("product_code").isNull() | F.col("customer_code").isNull() | F.col("date").isNull()
).count()
parent_duplicates = gold_parent_fact.groupBy(parent_keys).count().filter(F.col("count") > 1).count()
invalid_sales = gold_parent_fact.filter(F.col("sold_quantity") <= 0).count()

print(f"\n[AUDIT 2] Monthly Parent Fact Table (gold.fact_{data_source}):")
print(f"  ├─ Total Aggregated Records: {parent_total}")
print(f"  ├─ Null Key Components: {parent_nulls}")
print(f"  ├─ True Duplicate Aggregated Keys: {parent_duplicates}")
print(f"  └─ Invalid Sales Quantities (<= 0): {invalid_sales}")

if parent_duplicates > 0:
    print("WARNING: Duplicates detected in Monthly Parent Table!")
    gold_parent_fact.groupBy(parent_keys).count().filter(F.col("count") > 1).show(5)
else:
    print("PASSED: Monthly Parent Table is completely aggregated and unique.")

# Final Verdict
print("\n" + "=" * 70)
total_issues = sb_nulls + sb_duplicates + parent_nulls + parent_duplicates + invalid_sales
if total_issues == 0:
    print("FINAL RESULT: ALL GOLD LAYER TABLES ARE FULLY SANITIZED & PRODUCTION READY!")
else:
    print(f"FINAL RESULT: DETECTED {total_issues} ISSUES IN GOLD TABLES. REVIEW LOGS ABOVE.")
print("=" * 70)

 🚀 STARTING FINAL QUALITY AUDIT FOR GOLD LAYER TABLES

[AUDIT 1] Detailed Gold Fact Table (gold.sb_fact_orders):
  ├─ Total Gold Records: 40939
  ├─ Null Key Components: 0
  └─ True Duplicate Composite Keys: 128
  ⚠️ WARNING: Duplicates detected in Detailed Gold Table!
+-------------+--------------------+-------------+-----+
|     order_id|        product_code|customer_code|count|
+-------------+--------------------+-------------+-----+
| FJUL34603603|451f7167b28a25bde...|       789603|    2|
| FAUG47121401|e91ba9d665f90254d...|       789121|    2|
| FAUG48603103|102628255d24304d6...|       789603|    2|
|FAUG413403602|ac8b5d1e6a188f56b...|       789403|    2|
| FSEP55402603|da6bfc596c1360ca0...|       789402|    2|
+-------------+--------------------+-------------+-----+
only showing top 5 rows

[AUDIT 2] Monthly Parent Fact Table (gold.fact_orders):
  ├─ Total Aggregated Records: 96722
  ├─ Null Key Components: 0
  ├─ True Duplicate Aggregated Keys: 0
  └─ Invalid Sales Quantities (<